In [4]:
import spotipy
import os
from spotipy.oauth2 import SpotifyOAuth

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.getenv('spotify_client_ID'),
    client_secret=os.getenv('spotify_client_secret'),
    redirect_uri="http://127.0.0.1:3000",
    scope="user-top-read"
))

In [5]:
def get_top_artists(time_period, offset=0):
    all_collected_artists = []

    while True: 
        top_artists = sp.current_user_top_artists(limit=50, offset=offset, time_range=time_period)
        all_collected_artists += top_artists['items']
        if top_artists['next'] is None:
            break
        else:
            offset += 50

    return [artist['name'] for artist in all_collected_artists]

long_term_artists = get_top_artists("long_term")
short_term_artists = get_top_artists("short_term")
medium_term_artists = get_top_artists("medium_term")

In [6]:
def get_top_tracks(time_period, offset=0):
    all_collected_artists_and_tracks = []

    while True: 
        top_tracks = sp.current_user_top_tracks(limit=50, offset=offset, time_range=time_period)
        for track in top_tracks['items']:
            top_track_dict = {
                "artist_name": track['artists'][0]['name'],
                "track_name": track['name']
            }
            all_collected_artists_and_tracks.append(top_track_dict)
        if top_tracks['next'] is None:
            break
        else:
            offset += 50

    return all_collected_artists_and_tracks

#medium_term_artists_and_tracks = get_top_tracks("medium_term")
long_term_artists_and_tracks = get_top_tracks("long_term")
#short_term_artists_and_tracks = get_top_tracks("short_term")

### Feature Engineering from Spotify API #1: Time Range Consistency
- How many of the three time ranges (short, medium, long) does the artist appear in?
- An artist you listen to across short, medium, and long term is a sustained obsession, not a phase. Much stronger signal than appearing in just one window.

In [7]:
import pandas as pd
df_short = pd.DataFrame(short_term_artists, columns=["artist_name"])
df_medium = pd.DataFrame(medium_term_artists, columns=["artist_name"])
df_long = pd.DataFrame(long_term_artists, columns=["artist_name"])

df_short["in_short"] = 1
df_medium["in_medium"] = 1
df_long["in_long"] = 1

consistency = df_long.merge(df_medium, on="artist_name", how="outer") \
                     .merge(df_short, on="artist_name", how="outer") \
                     .fillna(0)

consistency["time_range_consistency"] = (consistency["in_short"] + consistency["in_medium"] + consistency["in_long"]) / 3

### Feature Engineering # 2: Presence (from top artists) in long term listening range
- Did this artist appear in your Spotify long-term top artists?
- Spotify's long-term top artists endpoint reflects actual engagement with hoe Spotify measures it — weighted by recency


In [10]:
total = len(long_term_artists)

api_presence = pd.DataFrame([
    {
        "artist_name": artist,
        "api_presence_score": (total - rank) / (total - 1)
    }
    for rank, artist in enumerate(long_term_artists, start=1)
])

### Feature Engineering #3 : Track Breadth (long-term)
- How many distinct songs from this artist appear in your long-term top tracks?
- Multiple songs in top tracks = deep engagement, not a one-hit obsession

In [13]:
df_tracks = pd.DataFrame(long_term_artists_and_tracks)
track_breadth = df_tracks.groupby("artist_name")["track_name"].nunique().reset_index()
track_breadth.columns = ["artist_name", "track_breadth"]
track_breadth["track_breadth_score"] = track_breadth["track_breadth"] / track_breadth["track_breadth"].max()

In [18]:
import duckdb as dd
conn = dd.connect('spotify_all_years.db')
df = conn.execute("SELECT * FROM superfan_scores").df()

In [34]:
df = conn.execute("SELECT * FROM superfan_scores").df()
df = df.merge(api_presence, on="artist_name", how="left")
df = df.merge(track_breadth[["artist_name", "track_breadth_score"]], on="artist_name", how="left")
df = df.merge(consistency[["artist_name", "time_range_consistency"]], on="artist_name", how="left")
df[["superfan_score", "api_presence_score", "time_range_consistency", "track_breadth_score"]] = df[["superfan_score", "api_presence_score", "time_range_consistency", "track_breadth_score"]].fillna(0)
df["blended_score"] = (0.7 * df["superfan_score"]) + (0.15 * df["api_presence_score"]) + (0.10 * df["time_range_consistency"]) + (0.05 * df["track_breadth_score"])

In [62]:
df["blended_score"] = (0.60 * df["superfan_score"]) + (0.20 * df["api_presence_score"]) + (0.10 * df["time_range_consistency"]) + (0.10 * df["track_breadth_score"])

In [63]:
df_final = df.sort_values("blended_score", ascending=False).head(50)

In [65]:
df_final[["artist_name", "superfan_score", "api_presence_score", "time_range_consistency", "track_breadth_score", "blended_score"]].to_csv("superfan_scores_blended.csv", index=False)